# Tugas Individu: Studi Kasus Klasifikasi
## Prediksi Gagal Bayar Kartu Kredit (*Credit Card Default Prediction*) dengan Machine Learning

**Nama:** Muhammad Faeruz Wafi Abidin  
**Program Studi:** D3 Statistika Terapan dan Komputasi, Universitas Negeri Semarang  
**Mata Kuliah:** *(isi nama mata kuliah)*  

---
**Daftar isi**
1. Permasalahan
2. Metode klasifikasi
3. Dataset
4. Tahapan penerapan (EDA, preprocessing, pemodelan, evaluasi)
5. Hasil dan pembahasan
6. Manfaat
7. Kesimpulan
8. Referensi

## 1. Permasalahan

Bank penerbit kartu kredit menghadapi risiko **gagal bayar (*default*)**, yaitu kondisi ketika nasabah tidak mampu membayar tagihan pada bulan berikutnya. Gagal bayar menyebabkan kerugian finansial dan memengaruhi kualitas portofolio kredit. Di sisi lain, jika bank terlalu ketat menolak nasabah, peluang pendapatan dari bunga dan biaya layanan ikut hilang.

Kasus ini dirujuk dari penelitian Yeh & Lien (2009) pada data nasabah kartu kredit di Taiwan. Pada periode tersebut, penerbitan kartu yang agresif membuat banyak nasabah memiliki utang melebihi kemampuan bayar, sehingga bank membutuhkan alat untuk menilai risiko nasabah secara objektif.

**Rumusan masalah:** apakah seorang nasabah akan **gagal bayar pada bulan berikutnya** (kelas 1) atau **tidak** (kelas 0), berdasarkan profil demografi, riwayat pembayaran, jumlah tagihan, dan jumlah pembayaran selama 6 bulan terakhir?

Ini adalah masalah **klasifikasi biner** dengan kelas yang **tidak seimbang** (nasabah gagal bayar adalah minoritas).

## 2. Metode Klasifikasi

Tiga model dibandingkan, dari yang sederhana ke yang lebih kompleks:

| Model | Alasan dipilih |
|---|---|
| **Logistic Regression** | *Baseline*; mudah diinterpretasi dan lazim dipakai pada *credit scoring*. |
| **Random Forest** (Breiman, 2001) | Kumpulan pohon keputusan (*bagging*); menangkap hubungan non-linear dan tahan terhadap *overfitting*. |
| **Gradient Boosting** | Pohon dibangun berurutan, tiap pohon memperbaiki kesalahan sebelumnya; biasanya akurat pada data tabular. |

Karena kelas tidak seimbang, **akurasi saja tidak memadai**. Evaluasi memakai *precision*, *recall*, *F1-score*, dan **ROC-AUC**. Untuk Logistic Regression dan Random Forest dipakai `class_weight="balanced"` agar kelas minoritas tidak diabaikan.

## 3. Dataset

**Default of Credit Card Clients** (UCI Machine Learning Repository), 30.000 nasabah kartu kredit di Taiwan, April sampai September 2005. Sumber: Yeh & Lien (2009).

| Kelompok | Variabel | Keterangan |
|---|---|---|
| Kredit | `LIMIT_BAL` | Batas kredit (NT$) |
| Demografi | `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` | Jenis kelamin, pendidikan, status pernikahan, usia |
| Riwayat bayar | `PAY_1` ... `PAY_6` | Status pembayaran bulan September sampai April (-1 = lunas tepat waktu, 1 = telat 1 bulan, dst.) |
| Tagihan | `BILL_AMT1` ... `BILL_AMT6` | Jumlah tagihan tiap bulan |
| Pembayaran | `PAY_AMT1` ... `PAY_AMT6` | Jumlah yang dibayar tiap bulan |
| **Target** | `DEFAULT` | 1 = gagal bayar bulan berikutnya, 0 = tidak |

Dataset: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

## 4. Tahapan Penerapan

### 4.1 Impor library dan memuat data

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# Muat dataset dari UCI (id=350). Jika gagal, pakai OpenML sebagai cadangan.
NAMES = (["LIMIT_BAL","SEX","EDUCATION","MARRIAGE","AGE"]
         + [f"PAY_{i}" for i in range(1,7)]
         + [f"BILL_AMT{i}" for i in range(1,7)]
         + [f"PAY_AMT{i}" for i in range(1,7)])

try:
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=350)
    X = ds.data.features.copy()
    y = ds.data.targets.iloc[:, 0].copy()
except Exception as e:
    print("UCI gagal, memakai OpenML:", e)
    from sklearn.datasets import fetch_openml
    ds = fetch_openml(data_id=42477, as_frame=True)
    X = ds.data.copy()
    y = ds.target.copy()

X.columns = NAMES            # kolom diurutkan sesuai dokumentasi dataset
y = y.astype(int).rename("DEFAULT")
df = pd.concat([X, y], axis=1)
print(df.shape)
df.head()

### 4.2 Eksplorasi data (EDA)

In [ ]:
df.info()
print("\nMissing value per kolom:", df.isna().sum().sum())
print("Baris duplikat:", df.duplicated().sum())
df.describe().T

In [ ]:
# Distribusi target
counts = df["DEFAULT"].value_counts().sort_index()
print(counts, "\nProporsi gagal bayar: {:.2%}".format(df["DEFAULT"].mean()))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x="DEFAULT", data=df, ax=ax[0]); ax[0].set_title("Distribusi kelas target")
sns.histplot(data=df, x="LIMIT_BAL", hue="DEFAULT", bins=40, element="step", ax=ax[1]); ax[1].set_title("Batas kredit")
sns.histplot(data=df, x="AGE", hue="DEFAULT", bins=30, element="step", ax=ax[2]); ax[2].set_title("Usia")
plt.tight_layout(); plt.show()

In [ ]:
# Tingkat gagal bayar menurut status pembayaran bulan terakhir (PAY_1)
rate = df.groupby("PAY_1")["DEFAULT"].agg(["mean", "count"]).reset_index()
plt.figure(figsize=(8,4))
sns.barplot(data=rate, x="PAY_1", y="mean", color="steelblue")
plt.ylabel("Proporsi gagal bayar"); plt.title("Gagal bayar vs status pembayaran terakhir")
plt.show()
rate

In [ ]:
plt.figure(figsize=(12,9))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Korelasi antar variabel"); plt.show()
print(df.corr()["DEFAULT"].drop("DEFAULT").sort_values(ascending=False).head(8))

### 4.3 Preprocessing

- Dokumentasi dataset hanya mendefinisikan `EDUCATION` 1 sampai 4 dan `MARRIAGE` 1 sampai 3, tetapi data memuat kode lain (0, 5, 6). Kode ini digabung ke kategori "lainnya".
- Data dibagi 80% latih dan 20% uji dengan **stratifikasi** agar proporsi kelas terjaga.
- Standardisasi fitur hanya dipakai untuk Logistic Regression. Model berbasis pohon tidak membutuhkannya.
- *Scaler* dipasang hanya pada data latih (di dalam `Pipeline`) untuk mencegah **kebocoran data**.

In [ ]:
print("EDUCATION sebelum:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE sebelum:", sorted(df["MARRIAGE"].unique()))

df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

print("EDUCATION sesudah:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE sesudah:", sorted(df["MARRIAGE"].unique()))

In [ ]:
X = df.drop(columns="DEFAULT")
y = df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Latih:", X_train.shape, "| Uji:", X_test.shape)
print("Proporsi gagal bayar latih: {:.3f} | uji: {:.3f}".format(y_train.mean(), y_test.mean()))

### 4.4 Pemodelan dan validasi silang

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced",
        n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_rows.append({"Model": name, "ROC-AUC (CV)": auc.mean(), "Std AUC": auc.std(), "F1 (CV)": f1.mean()})

pd.DataFrame(cv_rows).round(4)

### 4.5 Evaluasi pada data uji

In [ ]:
rows, fitted, probas = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    fitted[name], probas[name] = model, prob
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob),
    })
results = pd.DataFrame(rows).set_index("Model").round(4)
results

In [ ]:
# Kurva ROC dan Precision-Recall
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, prob in probas.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
    p, r, _ = precision_recall_curve(y_test, prob)
    ax[1].plot(r, p, label=name)
ax[0].plot([0,1],[0,1],"k--"); ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate"); ax[0].set_title("Kurva ROC"); ax[0].legend()
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Kurva Precision-Recall"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Model terbaik berdasarkan ROC-AUC data uji
best_name = results["ROC-AUC"].idxmax()
print("Model terbaik:", best_name)

pred_best = (probas[best_name] >= 0.5).astype(int)
print(classification_report(y_test, pred_best, target_names=["Tidak gagal bayar", "Gagal bayar"]))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_best),
                       display_labels=["Tidak", "Gagal bayar"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix: {best_name}"); plt.show()

### 4.6 Penyesuaian ambang batas (*threshold*)

Ambang 0,5 belum tentu optimal. Bagi bank, **gagal mendeteksi nasabah berisiko (*false negative*)** biasanya lebih mahal daripada salah menandai nasabah baik. Berikut dampak perubahan ambang terhadap *precision* dan *recall* pada model terbaik.

In [ ]:
prob = probas[best_name]
thr_rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pr = (prob >= t).astype(int)
    thr_rows.append({"Threshold": t,
                     "Precision": precision_score(y_test, pr),
                     "Recall": recall_score(y_test, pr),
                     "F1": f1_score(y_test, pr)})
pd.DataFrame(thr_rows).round(4)

### 4.7 Faktor yang paling berpengaruh (*permutation importance*)

In [ ]:
perm = permutation_importance(fitted[best_name], X_test, y_test,
                              scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
sns.barplot(x=imp.values, y=imp.index, color="steelblue")
plt.xlabel("Penurunan ROC-AUC saat fitur diacak"); plt.title(f"10 fitur terpenting ({best_name})")
plt.show()
imp.round(4)

## 5. Hasil dan Pembahasan

> **Catatan pengisian:** angka di bawah ini harus disesuaikan dengan output yang muncul setelah seluruh sel dijalankan (*Runtime > Run all*). Ganti bagian bertanda `[...]`.

**Perbandingan model.** Tabel pada bagian 4.5 menunjukkan model **[nama model terbaik]** memiliki ROC-AUC tertinggi sebesar **[...]**, dibandingkan Logistic Regression **[...]** dan model lainnya **[...]**. Akurasi berada di kisaran **[...]**, tetapi karena sekitar 22% nasabah adalah gagal bayar, model yang selalu menebak "tidak gagal bayar" saja sudah mencapai akurasi sekitar 78%. Karena itu *recall* dan ROC-AUC lebih bermakna.

**Kinerja pada kelas gagal bayar.** Pada ambang 0,5, model terbaik memiliki *precision* **[...]** dan *recall* **[...]** untuk kelas gagal bayar. Artinya, dari seluruh nasabah yang benar-benar gagal bayar, sekitar **[...]%** berhasil terdeteksi. Pada bagian 4.6, menurunkan ambang meningkatkan *recall* dengan konsekuensi turunnya *precision*. Pemilihan ambang bergantung pada biaya kesalahan menurut kebijakan bank.

**Faktor penentu.** Pada bagian 4.7, variabel **[...]** muncul paling berpengaruh. Pola ini sejalan dengan analisis EDA: nasabah yang sudah menunggak pada bulan terakhir (`PAY_1` bernilai positif) memiliki proporsi gagal bayar jauh lebih tinggi daripada nasabah yang membayar tepat waktu.

**Keterbatasan.**
- Data berasal dari satu bank di Taiwan tahun 2005, sehingga belum tentu mewakili kondisi lain.
- Hyperparameter belum dioptimasi secara menyeluruh (misalnya dengan `GridSearchCV`).
- Hasil menunjukkan asosiasi prediktif, bukan hubungan sebab-akibat.
- Variabel demografi seperti jenis kelamin perlu dipertimbangkan dari sisi etika dan regulasi jika dipakai dalam keputusan kredit nyata.

## 6. Manfaat

1. **Manajemen risiko kredit.** Bank dapat mengidentifikasi nasabah berisiko tinggi lebih awal dan mengambil tindakan pencegahan, misalnya pengingat pembayaran atau restrukturisasi.
2. **Penentuan limit kredit.** Skor risiko membantu bank menetapkan atau menyesuaikan batas kredit berdasarkan profil nasabah.
3. **Efisiensi operasional.** Penilaian otomatis menggantikan sebagian proses manual, dan tim penagihan dapat memprioritaskan nasabah dengan skor risiko tertinggi.
4. **Pengambilan keputusan berbasis data.** Faktor risiko yang teridentifikasi (misalnya riwayat keterlambatan) dapat menjadi dasar kebijakan penerbitan kartu.
5. **Manfaat bagi nasabah.** Nasabah yang berisiko kesulitan bayar dapat dihubungi lebih dini sehingga tidak terjerat tunggakan berkepanjangan.

## 7. Kesimpulan

Masalah prediksi gagal bayar kartu kredit dapat dirumuskan sebagai klasifikasi biner dengan kelas tidak seimbang. Dengan tahapan EDA, preprocessing, pemodelan dengan validasi silang, dan evaluasi memakai metrik yang sesuai (*recall*, F1, ROC-AUC), model Machine Learning mampu membedakan nasabah berisiko dan tidak berisiko secara lebih baik daripada tebakan acak. Pemilihan ambang batas perlu disesuaikan dengan biaya kesalahan di dunia nyata. Untuk penerapan sesungguhnya, model perlu divalidasi pada data yang lebih mutakhir, dipantau secara berkala, dan dikaji dari sisi keadilan serta kepatuhan regulasi.

## 8. Referensi

1. Yeh, I-C., & Lien, C-H. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. *Expert Systems with Applications*, 36(2), 2473-2480. https://doi.org/10.1016/j.eswa.2007.12.020
2. Yeh, I-C. (2016). *Default of Credit Card Clients* [Dataset]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
3. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.
4. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32.
5. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
6. Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861-874.